Python GPU Programming with CuPy
=================================

CuPy is a NumPy-compatible library that runs on NVIDIA GPUs.
It's the easiest way to accelerate NumPy code!

Installation:
- Google Colab: Already installed! Just import it.
- Local: pip install cupy-cuda11x (replace 11x with your CUDA version)

For CPU testing (no GPU required):
- We'll show both CuPy and NumPy versions

In [2]:
import numpy as np
import time

# Try to import CuPy
try:
    import cupy as cp
    CUPY_AVAILABLE = True
    print(f"CuPy available! Device: {cp.cuda.Device()}")
except ImportError:
    print("CuPy not available. Results will only show NumPy.")
    CUPY_AVAILABLE = False

CuPy available! Device: <CUDA Device 0>


# ============================================================================
# EXAMPLE 1: CuPy Basics - Drop-in NumPy Replacement
# ============================================================================


In [3]:
def example1_basics():
    print("\n" + "="*60)
    print("EXAMPLE 1: CuPy Basics")
    print("="*60)

    # NumPy arrays (CPU)
    np_array = np.array([1, 2, 3, 4, 5])
    print(f"NumPy array: {np_array}")
    print(f"   Type: {type(np_array)}")
    print(f"   Location: CPU RAM")

    if CUPY_AVAILABLE:
        # CuPy arrays (GPU)
        cp_array = cp.array([1, 2, 3, 4, 5])
        print(f"\nCuPy array: {cp_array}")
        print(f"   Type: {type(cp_array)}")
        print(f"   Location: GPU Memory")

        # Converting between NumPy and CuPy
        print("\nMoving data between CPU and GPU:")
        cpu_data = np.array([1, 2, 3])
        gpu_data = cp.asarray(cpu_data)  # CPU → GPU
        print(f"   CPU → GPU: {gpu_data}")

        back_to_cpu = cp.asnumpy(gpu_data)  # GPU → CPU
        print(f"   GPU → CPU: {back_to_cpu}")


# ============================================================================
# EXAMPLE 2: Performance Comparison
# ============================================================================


In [4]:

def benchmark(func, *args, name="", warmup=True):
    """Utility function to benchmark GPU/CPU code"""
    if warmup and CUPY_AVAILABLE:
        func(*args)  # Warm up
        if any(isinstance(arg, cp.ndarray) for arg in args):
            cp.cuda.Stream.null.synchronize()

    start = time.time()
    result = func(*args)
    if CUPY_AVAILABLE and any(isinstance(arg, cp.ndarray) for arg in args):
        cp.cuda.Stream.null.synchronize()  # Wait for GPU
    elapsed = time.time() - start

    print(f"   {name:20s}: {elapsed*1000:8.2f} ms")
    return result, elapsed

def example2_performance():
    """
    Small data: GPU is slower (overhead from data transfer + kernel launch)
    Large data: GPU is much faster (parallelism wins over overhead)
    """

    print("\n" + "="*60)
    print("EXAMPLE 2: Performance Comparison")
    print("="*60)

    # Test different sizes
    sizes = [1000, 10_000, 100_000, 1_000_000]

    for n in sizes:
        print(f"\n📊 Array size: {n:,} elements")

        # Create data
        np_a = np.random.rand(n).astype(np.float32)
        np_b = np.random.rand(n).astype(np.float32)

        # NumPy (CPU)
        result_cpu, cpu_time = benchmark(
            lambda a, b: a**2 + b**2 + 2*a*b,
            np_a, np_b,
            name="NumPy (CPU)"
        )

        if CUPY_AVAILABLE:
            # CuPy (GPU)
            cp_a = cp.asarray(np_a)
            cp_b = cp.asarray(np_b)

            result_gpu, gpu_time = benchmark(
                lambda a, b: a**2 + b**2 + 2*a*b,
                cp_a, cp_b,
                name="CuPy (GPU)"
            )

            speedup = cpu_time / gpu_time
            print(f"   {'Speedup':20s}: {speedup:8.2f}x")

            # Verify correctness
            result_gpu_cpu = cp.asnumpy(result_gpu)
            matches = np.allclose(result_cpu, result_gpu_cpu)
            print(f"   Results match: {matches}")



# ============================================================================
# EXAMPLE 3: Matrix Operations
# ============================================================================


In [5]:

def example3_matrix_operations():
    """
    Why GPUs excel here:

    - Matrix multiplication has O(n³) operations but only O(n²) data
    - Each element of C can be computed independently → massive parallelism
    - GPUs can compute thousands of elements simultaneously

    Matrix multiplication is the "killer app" for GPUs - expect 20-100x speedups on large matrices!
    This is why GPUs dominate machine learning
    (neural networks are just giant matrix multiplications!!!~).

    """


    print("\n" + "="*60)
    print("EXAMPLE 3: Matrix Operations (Where GPU Shines!)")
    print("="*60)

    sizes = [100, 500, 1000, 2000]

    for n in sizes:
        print(f"\n📊 Matrix size: {n}×{n}")

        # Create matrices
        np_a = np.random.rand(n, n).astype(np.float32)
        np_b = np.random.rand(n, n).astype(np.float32)

        # NumPy matrix multiplication
        result_cpu, cpu_time = benchmark(
            np.dot, np_a, np_b,
            name="NumPy matmul",
            warmup=False
        )

        if CUPY_AVAILABLE:
            # CuPy matrix multiplication
            cp_a = cp.asarray(np_a)
            cp_b = cp.asarray(np_b)

            result_gpu, gpu_time = benchmark(
                cp.dot, cp_a, cp_b,
                name="CuPy matmul"
            )

            speedup = cpu_time / gpu_time
            print(f"   {'Speedup':20s}: {speedup:8.2f}x {'🚀' if speedup > 10 else ''}")

            # For large matrices, verify only a sample
            if n <= 500:
                result_gpu_cpu = cp.asnumpy(result_gpu)
                matches = np.allclose(result_cpu, result_gpu_cpu, rtol=1e-4)
                print(f"   Results match: {matches}")


# ============================================================================
# EXAMPLE 4: Life Sciences Application - Image Filtering
# ============================================================================


In [6]:

def gaussian_blur(img, sigma=2.0):
    """Simple Gaussian blur filter"""
    from scipy.ndimage import gaussian_filter
    return gaussian_filter(img, sigma=sigma)

def example4_image_processing():
    """
    The operation: 3×3 Box Blur:
    What it does: For each pixel, replace it with the average of its 3×3 neighborhood:

Original image:        After blur:
┌─────────┐           ┌─────────┐
│ 5 8 3   │           │         │
│ 2 [7] 9 │  →        │   [6]   │  (average of 9 values)
│ 4 1 6   │           │         │
└─────────┘           └─────────┘

    Example fo rewal world application:
    100 CT slices per scan -> With GPU: 15 seconds vs 4+ minutes on CPU
    This is why NVIDIA GPUs are everywhere in hospitals and research labs!
    """
    print("\n" + "="*60)
    print("EXAMPLE 4: Image Processing (Medical Imaging)")
    print("="*60)

    # Simulate a medical image (e.g., CT scan slice)
    sizes = [(512, 512), (1024, 1024), (2048, 2048)]

    for size in sizes:
        print(f"\n📊 Image size: {size[0]}×{size[1]} ({size[0]*size[1]:,} pixels)")

        # Create synthetic medical image
        np_image = np.random.rand(*size).astype(np.float32)

        # CPU processing
        print("   Processing on CPU...")
        start = time.time()
        # Simple blur operation
        result_cpu = np.zeros_like(np_image)
        for i in range(1, size[0]-1):
            for j in range(1, size[1]-1):
                result_cpu[i, j] = (
                    np_image[i-1:i+2, j-1:j+2].mean()
                )
        cpu_time = time.time() - start
        print(f"   CPU time: {cpu_time*1000:.2f} ms")

        if CUPY_AVAILABLE:
            # GPU processing
            cp_image = cp.asarray(np_image)
            print("   Processing on GPU...")

            # Warm up
            cp.asnumpy(cp_image)

            start = time.time()
            # Use CuPy's convolution (highly optimized)
            # Even though the loop syntax looks the same, CuPy's operations are GPU-accelerated!!!
            kernel = cp.ones((3, 3), dtype=cp.float32) / 9.0
            result_gpu = cp.zeros_like(cp_image)
            for i in range(1, size[0]-1):
                for j in range(1, size[1]-1):
                    result_gpu[i, j] = (
                        cp_image[i-1:i+2, j-1:j+2].mean()
                    )
            cp.cuda.Stream.null.synchronize()
            gpu_time = time.time() - start

            print(f"   GPU time: {gpu_time*1000:.2f} ms")
            print(f"   Speedup: {cpu_time/gpu_time:.2f}x")



# ============================================================================
# EXAMPLE 5: Memory Transfer Overhead
# ============================================================================


In [7]:
# This function teaches the critical lesson about memory transfer overhead -
# one of the biggest pitfalls in GPU programming!

#❌ BAD - Transfer every iteration:
#for i in range(1000):
#    gpu_data = cp.asarray(cpu_data)  # Transfer
#    result = process(gpu_data)        # Compute
#    cpu_result = cp.asnumpy(result)  # Transfer back
## 2000 transfers! (1000 uploads + 1000 downloads)

#✅ GOOD - Keep data on GPU:
#gpu_data = cp.asarray(cpu_data)  # Transfer ONCE
#for i in range(1000):
#    gpu_data = process(gpu_data)  # Stay on GPU!
#cpu_result = cp.asnumpy(gpu_data)  # Transfer ONCE
## Only 2 transfers total!


def example5_memory_transfer():
    """
CPU RAM          PCIe Bus          GPU Memory
────────         ──────────         ──────────
[data]    ──────────────────►                    Phase 1: Transfer TO (slow)
                                    [data]
                                    ↓
                                  [compute]      Phase 2: Compute (FAST!)
                                    ↓
                                  [result]
          ◄──────────────────      [result]     Phase 3: Transfer FROM (slow)
[result]

   CPU → GPU: 12.50 ms     ← Slow!
   GPU compute: 1.20 ms    ← Fast!
   GPU → CPU: 11.80 ms     ← Slow!
   Total time: 25.50 ms
   Transfer overhead: 95.3%  ← 95% of time wasted on transfers!

   The shocking truth:
   Computation: 1.2 ms (5% of time) -> Only 5% of the time was spent actually computing!!!!
   Data movement: 24.3 ms (95% of time)

   Memory transfers can dominate! Keep data on GPU when possible.
    """

    print("\n" + "="*60)
    print("EXAMPLE 5: Understanding Memory Transfer Overhead")
    print("="*60)

    if not CUPY_AVAILABLE:
        print("CuPy required for this example")
        return

    n = 10_000_000
    print(f"Array size: {n:,} elements ({n*4/1024/1024:.2f} MB)")

    # Create CPU data
    np_data = np.random.rand(n).astype(np.float32)

    # Measure transfer times
    print("\n⏱️  Timing breakdown:")

    # CPU → GPU transfer
    start = time.time()
    cp_data = cp.asarray(np_data)
    cp.cuda.Stream.null.synchronize()
    transfer_to_gpu = time.time() - start
    print(f"   CPU → GPU: {transfer_to_gpu*1000:.2f} ms")

    # GPU computation
    start = time.time()
    result_gpu = cp.sqrt(cp_data**2 + 1)
    cp.cuda.Stream.null.synchronize()
    compute_time = time.time() - start
    print(f"   GPU compute: {compute_time*1000:.2f} ms")

    # GPU → CPU transfer
    start = time.time()
    result_cpu = cp.asnumpy(result_gpu)
    transfer_to_cpu = time.time() - start
    print(f"   GPU → CPU: {transfer_to_cpu*1000:.2f} ms")

    # Total
    total_time = transfer_to_gpu + compute_time + transfer_to_cpu
    print(f"\n   Total time: {total_time*1000:.2f} ms")
    print(f"   Transfer overhead: {((transfer_to_gpu + transfer_to_cpu)/total_time)*100:.1f}%")

    print("\n💡 Key insight: Memory transfers can dominate! Keep data on GPU when possible.")



# ============================================================================
# MAIN PROGRAM
# ============================================================================


In [8]:

def main():
    print("Python GPU Programming with CuPy")
    print("=" * 60)

    example1_basics()
    example2_performance()
    example3_matrix_operations()
    example4_image_processing()
    example5_memory_transfer()

    print("\n" + "="*60)
    print("CuPy Tutorial Complete!")
    print("="*60)
    print("\n💡 Key Takeaways:")
    print("   1. CuPy is a drop-in replacement for NumPy")
    print("   2. GPU excels at large array operations")
    print("   3. Matrix operations see huge speedups (10-100x)")
    print("   4. Memory transfer has overhead - minimize it!")
    print("   5. Keep data on GPU for multiple operations")

    print("\n✨ Pro Tips:")
    print("   - Use cp.asarray() once at the start")
    print("   - Do all processing on GPU")
    print("   - Use cp.asnumpy() once at the end")
    print("   - Avoid CPU ↔ GPU transfers in loops!")

if __name__ == "__main__":
    main()


Python GPU Programming with CuPy

EXAMPLE 1: CuPy Basics
NumPy array: [1 2 3 4 5]
   Type: <class 'numpy.ndarray'>
   Location: CPU RAM

CuPy array: [1 2 3 4 5]
   Type: <class 'cupy.ndarray'>
   Location: GPU Memory

Moving data between CPU and GPU:
   CPU → GPU: [1 2 3]
   GPU → CPU: [1 2 3]

EXAMPLE 2: Performance Comparison

📊 Array size: 1,000 elements
   NumPy (CPU)         :     0.01 ms
   CuPy (GPU)          :     0.24 ms
   Speedup             :     0.04x
   Results match: True

📊 Array size: 10,000 elements
   NumPy (CPU)         :     0.02 ms
   CuPy (GPU)          :     0.16 ms
   Speedup             :     0.14x
   Results match: True

📊 Array size: 100,000 elements
   NumPy (CPU)         :     0.16 ms
   CuPy (GPU)          :     0.17 ms
   Speedup             :     0.97x
   Results match: True

📊 Array size: 1,000,000 elements
   NumPy (CPU)         :     2.03 ms
   CuPy (GPU)          :     0.40 ms
   Speedup             :     5.07x
   Results match: True

EXAMPLE 3: Mat